# 01 - Reanálise reproduzível
Executa a auditoria CPU, os testes e os gráficos. Não treina um LLM.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "revisao_2026", cwd.parent / "revisao_2026",
              cwd / "cot-obfuscation-h2a" / "revisao_2026"]
ROOT = next((p for p in candidates if (p / "src/instruments.py").is_file()), None)
if ROOT is None:
    clone = cwd / "cot-obfuscation-h2a"
    if not clone.exists():
        subprocess.run(["git", "clone", "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git",
                        str(clone)], check=True)
    ROOT = clone / "revisao_2026"
assert (ROOT / "src/instruments.py").is_file(), ROOT
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
ENV = dict(os.environ, PYTHONPATH=str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""))
print("Raiz:", ROOT)


In [ ]:
INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.lock.txt"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], env=ENV, check=True)
subprocess.run([sys.executable, "src/reanalyse.py"], env=ENV, check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(ROOT / "results/summary.csv"))
display(pd.read_csv(ROOT / "results/contrasts.csv"))
display(pd.read_csv(ROOT / "results/grading_stress_summary.csv"))


Os resultados são reprocessamento de métricas depositadas e testes mecânicos. As 540 saídas integrais não estão no CSV original.
